# Real-Time Speech Captioning with Sarvam's Streaming STT (Saaras v3)

Live captions for a lecture, meeting, or video call: a mic feeds audio in, captions come out a few hundred milliseconds later. This recipe builds that pipeline end to end, and checks a few claims empirically along the way instead of assuming them.

**Correction up front:** Bulbul is Sarvam's *text-to-speech* model, not the speech-to-text one. Speech-to-text is a separate model family -- Saarika for batch (upload a file, get a transcript back) and **Saaras** for real-time streaming over WebSocket. This recipe uses Saaras.

Pipeline:

```
mic / audio source
      |
      v
resample to 16kHz mono PCM   <- the step people skip, then wonder why transcription is poor
      |
      v
wss://api.sarvam.ai/speech-to-text/ws   (model=saaras:v3)
      |
      v
captions rendered as they arrive
```

A notebook can't hold open a live browser microphone, so this recipe instead:

1. Synthesizes two test clips with Sarvam's TTS (Bulbul) -- one plain, one Hinglish (code-switched) -- so the whole pipeline runs without you sourcing your own audio.
2. Resamples them to 16kHz mono PCM, the one format the streaming socket accepts.
3. Streams them to Saaras v3 in small chunks, paced in real time, and prints captions as they arrive.
4. Empirically checks two easy-to-assume claims: whether `mode="codemix"`/`"translit"` actually handle code-switching, and whether the response ever populates a diarized transcript.
5. Implements reconnect-with-backoff for when the socket drops mid-session.
6. Points at [`examples/Live_Video_Transcription`](../Live_Video_Transcription) for the full browser-microphone version of this same pipeline.

In [ ]:
%pip install -r requirements.txt

## Setup

In [ ]:
from __future__ import annotations

import asyncio
import base64
import io
import json
import os
import time
from math import gcd
from pathlib import Path

import numpy as np
import soundfile as sf
from dotenv import load_dotenv
from IPython.display import Audio, display
from sarvamai import AsyncSarvamAI, SarvamAI
from sarvamai.play import save
from scipy.signal import resample_poly

load_dotenv()

SARVAM_API_KEY = os.getenv("SARVAM_API_KEY")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "Set SARVAM_API_KEY in your environment or .env file before running."
    )

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)
async_client = AsyncSarvamAI(api_subscription_key=SARVAM_API_KEY)

SAMPLE_DIR = Path("sample_data")
OUTPUT_DIR = Path("outputs")
SAMPLE_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. The model lineup

| Model | Task | Transport |
|---|---|---|
| **Bulbul** (`bulbul:v3`) | Text-to-speech | REST |
| **Saarika** (`saarika:v2.5`) | Speech-to-text, batch (upload a file) | REST |
| **Saaras** (`saaras:v3`) | Speech-to-text, real-time streaming | WebSocket |

This recipe only needs Saaras, used below via `client.speech_to_text_streaming.connect(...)`, which opens a socket at `wss://api.sarvam.ai/speech-to-text/ws`. Bulbul shows up too, but only to generate test audio so the rest of the notebook is runnable without you supplying your own recordings.

## 2. Generate test audio with Bulbul TTS

Two clips: a plain Hindi sentence with a phone number (mirrors the example in Sarvam's own streaming docs, so the `mode` comparisons later are easy to eyeball), and a Hinglish, code-switched sentence to actually test code-switching behavior later instead of assuming it.

In [ ]:
def synthesize(text: str, filename: str, language_code: str = "hi-IN") -> Path:
    response = client.text_to_speech.convert(
        text=text,
        target_language_code=language_code,
        speaker="shubh",
        model="bulbul:v3",
    )
    path = SAMPLE_DIR / filename
    save(response, str(path))
    return path


hindi_clip = synthesize("मेरा फोन नंबर है 9840950950", "lecture_hindi.wav")
hinglish_clip = synthesize(
    "आज हम machine learning का introduction cover करेंगे, फिर एक practical example देखेंगे।",
    "lecture_hinglish.wav",
)

display(Audio(str(hindi_clip)))
display(Audio(str(hinglish_clip)))

for clip in (hindi_clip, hinglish_clip):
    info = sf.info(clip)
    print(f"{clip.name}: {info.samplerate} Hz, {info.channels} channel(s), {info.duration:.2f}s")

## 3. Getting audio into the shape Saaras expects

The streaming endpoint only accepts WAV or raw PCM, and PCM is only supported at 16kHz. Browser mics typically capture at 44.1kHz or 48kHz -- and, as the cell above just showed, Bulbul's own default output isn't 16kHz either. Resampling to 16kHz mono before anything hits the socket is mandatory, not a nice-to-have.

In [ ]:
def resample_to_16k_mono_pcm16(src_path: Path, target_sr: int = 16000) -> Path:
    audio, sr = sf.read(src_path, always_2d=True)
    mono = audio.mean(axis=1)
    if sr != target_sr:
        g = gcd(sr, target_sr)
        mono = resample_poly(mono, target_sr // g, sr // g)
    pcm16 = np.clip(mono * 32767.0, -32768, 32767).astype(np.int16)
    dst_path = src_path.with_name(src_path.stem + "_16k.wav")
    sf.write(dst_path, pcm16, target_sr, subtype="PCM_16")
    return dst_path


hindi_16k = resample_to_16k_mono_pcm16(hindi_clip)
hinglish_16k = resample_to_16k_mono_pcm16(hinglish_clip)

for original, resampled in ((hindi_clip, hindi_16k), (hinglish_clip, hinglish_16k)):
    before = sf.info(original)
    after = sf.info(resampled)
    print(
        f"{original.name}: {before.samplerate} Hz, {before.channels} ch "
        f"-> {resampled.name}: {after.samplerate} Hz, {after.channels} ch"
    )

## 4. Chunking audio to simulate a live mic feed

A browser pushes small frames as they're captured, not the whole recording at once -- that's what makes captions feel live. Each frame below is a small, self-contained WAV file, matching what the SDK's `transcribe(audio=...)` call expects per message (default `encoding="audio/wav"`, `sample_rate=16000`).

In [ ]:
def iter_audio_frames(path: Path, frame_ms: int = 200):
    """Yield (base64_wav_str, duration_seconds) frames, one per `frame_ms` of audio."""
    samples, sr = sf.read(path, dtype="int16", always_2d=False)
    frame_len = int(sr * frame_ms / 1000)
    for start in range(0, len(samples), frame_len):
        chunk = samples[start : start + frame_len]
        if len(chunk) == 0:
            continue
        buf = io.BytesIO()
        sf.write(buf, chunk, sr, format="WAV", subtype="PCM_16")
        yield base64.b64encode(buf.getvalue()).decode("ascii"), len(chunk) / sr

## 5. Connecting to Saaras v3 and rendering captions live

Frames are sent paced in real time (`asyncio.sleep` between chunks, like a live mic would arrive), while a receiver task prints each transcript as it comes back. Each response carries real `audio_duration`/`processing_latency` numbers -- logged here instead of just quoting "milliseconds" from the docs. `ws.flush()` forces out the final partial segment once the audio ends.

In [ ]:
async def stream_captions(path: Path, mode: str = "transcribe", language_code: str = "hi-IN") -> list[str]:
    """Stream a 16kHz mono WAV file to Saaras v3 in real time, printing captions as they arrive."""
    captions: list[str] = []
    start = time.monotonic()

    async def sender(ws):
        for frame_b64, duration in iter_audio_frames(path):
            await ws.transcribe(audio=frame_b64)
            await asyncio.sleep(duration)
        await ws.flush()
        await asyncio.sleep(1.5)  # grace period for the final segment to come back

    async def receiver(ws):
        while True:
            resp = await ws.recv()
            if resp.type == "error":
                raise RuntimeError(f"Stream error {resp.data.code}: {resp.data.error}")
            if resp.type != "data" or not hasattr(resp.data, "transcript"):
                continue
            text = resp.data.transcript.strip()
            if not text:
                continue
            elapsed = time.monotonic() - start
            latency_ms = resp.data.metrics.processing_latency * 1000
            print(f"[{elapsed:5.2f}s, latency {latency_ms:5.0f}ms] {text}")
            captions.append(text)

    async with async_client.speech_to_text_streaming.connect(
        language_code=language_code, model="saaras:v3", mode=mode
    ) as ws:
        sender_task = asyncio.create_task(sender(ws))
        receiver_task = asyncio.create_task(receiver(ws))
        done, pending = await asyncio.wait(
            {sender_task, receiver_task}, return_when=asyncio.FIRST_COMPLETED
        )
        for task in pending:
            task.cancel()
        await asyncio.gather(*pending, return_exceptions=True)
        for task in done:
            task.result()  # re-raise if sender or receiver failed

    return captions

In [ ]:
transcript_hi = await stream_captions(hindi_16k, mode="transcribe", language_code="hi-IN")

## 6. `mode="translate"` -- captions in English from Indic speech

This is the hidden feature: no separate translation call. A Tamil-speaking lecturer's audio goes in, English captions come out, live -- same socket, same `stream_captions` function, one parameter different.

In [ ]:
translation = await stream_captions(hindi_16k, mode="translate", language_code="hi-IN")

## 7. Testing code-switching directly, instead of assuming

Sarvam's TTS docs confirm Bulbul handles code-mixed text like Hinglish. Whether the *streaming STT* model preserves code-switching mid-utterance is a separate question -- and the SDK actually exposes an answer to it directly: `mode="codemix"` keeps English words in English and Indic words in native script, and `mode="translit"` romanizes everything. Run the Hinglish clip through all three and compare, rather than picking one and hoping.

In [ ]:
results: dict[str, list[str]] = {}
for mode in ("transcribe", "codemix", "translit"):
    print(f"\n--- mode={mode} ---")
    results[mode] = await stream_captions(hinglish_16k, mode=mode, language_code="hi-IN")

print("\nSide by side:")
for mode, caps in results.items():
    print(f"{mode:10s}: {' '.join(caps)}")

In [ ]:
summary = {
    "transcribe": transcript_hi,
    "translate": translation,
    "codemix_comparison": results,
}
summary_path = OUTPUT_DIR / "streaming_captions_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(f"Saved summary to {summary_path}")

## 8. Checking diarization directly, instead of assuming

The response schema (`SpeechToTextTranscriptionData`) does carry an optional `diarized_transcript` field. That doesn't mean it comes back populated for every account, plan, or audio input -- check it against your own audio before designing a multi-speaker classroom or meeting feature around it. Until it's confirmed populated for your setup, the safer design is one mic (or one channel) per speaker.

In [ ]:
async def check_diarization(path: Path, max_messages: int = 20) -> None:
    async with async_client.speech_to_text_streaming.connect(
        language_code="hi-IN", model="saaras:v3", mode="transcribe"
    ) as ws:
        for frame_b64, _ in iter_audio_frames(path):
            await ws.transcribe(audio=frame_b64)
        await ws.flush()

        for _ in range(max_messages):
            try:
                resp = await asyncio.wait_for(ws.recv(), timeout=2.0)
            except asyncio.TimeoutError:
                break
            if resp.type == "data" and hasattr(resp.data, "transcript"):
                print("transcript:", resp.data.transcript)
                print("diarized_transcript:", resp.data.diarized_transcript)
                return
    print("No transcript message received to inspect.")


await check_diarization(hindi_16k)

## 9. Reconnecting when the socket drops

Two different error channels exist. In-band `{"type": "error"}` messages arrive over a socket that's still open -- `receiver` above already turns those into a raised exception. Separately, the socket itself can close: standard WebSocket close codes (1000-1015) are routine (idle timeouts, network blips) and worth a reconnect; codes 4000-4999 are Sarvam-specific application errors, and the docs are explicit that you should read the close reason string -- auth/quota failures need a fix, not a retry, while other 4xxx codes are safe to retry with backoff.

In [ ]:
import random

from websockets.exceptions import ConnectionClosed


async def stream_captions_with_backoff(
    path: Path,
    mode: str = "transcribe",
    language_code: str = "hi-IN",
    max_retries: int = 5,
) -> list[str]:
    """Like stream_captions, but reconnects on transient WebSocket drops."""
    attempt = 0
    while True:
        try:
            return await stream_captions(path, mode=mode, language_code=language_code)
        except ConnectionClosed as exc:
            code_, reason = exc.code, exc.reason or ""
            attempt += 1

            if 4000 <= code_ <= 4999:
                if any(word in reason.lower() for word in ("auth", "quota", "key", "unauthorized")):
                    raise RuntimeError(f"Non-retryable stream error {code_}: {reason}") from exc
                print(f"App error {code_} ({reason}), retrying...")
            elif 1000 <= code_ <= 1015:
                print(f"Connection closed ({code_} {reason}), reconnecting...")
            else:
                raise

            if attempt > max_retries:
                raise RuntimeError(f"Gave up after {max_retries} retries") from exc

            backoff = min(2**attempt, 30) + random.uniform(0, 1)
            await asyncio.sleep(backoff)

## 10. Taking this to the browser

Everything above proves out the Saaras v3 pipeline end to end, but a live captioning feature usually needs a real microphone in a real browser tab. Two things change:

- **The API key must never reach the browser.** Run a small backend (Flask, FastAPI, Node -- anything that can hold a WebSocket open) that authenticates to Sarvam and relays audio frames both ways. The browser talks to *your* backend, not directly to `wss://api.sarvam.ai`.
- **The browser does the resampling**, via the Web Audio API (an `AudioWorklet` reading the mic at 44.1/48kHz and downsampling to 16kHz mono), then pushes frames to your backend over its own socket -- the same shape of frame `iter_audio_frames` builds above, just produced client-side instead of from a file.

This repo has two complete, runnable versions of that:

- **`app.py`, right in this directory**: a Flask + Socket.IO app with a video-upload and a live-webcam tab, a language/captions-style picker, and a YouTube-style caption overlay. It's built on **`saaras:v3-realtime`** -- a newer, currently-beta streaming model with a different wire protocol from the `saaras:v3` this notebook uses -- connected to directly via `websockets` rather than through the SDK, since the SDK doesn't wrap the realtime endpoint yet. See this directory's `README.md` for the beta-access requirement and the architecture notes.
- [`examples/Live_Video_Transcription`](../Live_Video_Transcription): a Flask + Socket.IO app on the `saaras:v3` model this notebook uses, running persistent `transcribe` and `translate` streaming sessions concurrently -- the same `mode` parameter used throughout this notebook. Point its audio source at a live mic instead of a video element and the rest of the pipeline is unchanged.

## Summary -- confirmed vs. verify against your own account

**Confirmed by the SDK and exercised directly above:**
- Saaras v3 is the streaming STT model, reached at `wss://api.sarvam.ai/speech-to-text/ws` via `client.speech_to_text_streaming.connect(...)`.
- `mode` controls output shape: `transcribe`, `translate`, `verbatim`, `translit`, `codemix` -- the last two are a direct, built-in answer to code-switching.
- The socket only takes WAV or raw PCM, and PCM only at 16kHz -- resampling (from a browser mic, or from Bulbul's 24kHz default output, as seen above) is mandatory.
- Each response carries real `audio_duration`/`processing_latency` numbers to log, instead of relying on the docs' "milliseconds" framing.
- The response schema has a `diarized_transcript` field -- whether it comes back populated depends on your plan/account, which section 8 checks directly rather than assuming.

**Still worth testing against your own audio before shipping:**
- Whether `codemix`/`translit` hold up on your actual code-switching patterns (regional slang, domain jargon), not just the one example sentence here.
- Close-code behavior under real network conditions (idle timeouts, proxy drops) -- the reconnect helper above is a starting point, not a guarantee it covers every 4xxx code Sarvam may introduce.